<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/OptionsVolumeGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import csv

def generate_liquidity_eagle_universe():
    # File configuration
    file_notional = "OptionVolume200_Notional_20260501.csv"
    file_count = "OptionVolume200_Count_20260501.csv"
    output_file = "OptionVolume.csv"

    try:
        # Load datasets
        df_n = pd.read_csv(file_notional)
        df_c = pd.read_csv(file_count)

        # 1. Enforce $25 Price Floor (Universal Standard)
        df_n = df_n[df_n['Price'] >= 25].copy()
        df_c = df_c[df_c['Price'] >= 25].copy()

        # 2. Assign Weights (Rank 200 for top of each list)
        df_n = df_n.reset_index(drop=True)
        df_c = df_c.reset_index(drop=True)

        df_n['rn_val'] = 200 - df_n.index
        df_c['rc_val'] = 200 - df_c.index

        # 3. UNIFIED MERGE: Use 'outer' to capture symbols like DRAM (Count-only)
        df_n.set_index('Symbol', inplace=True)
        df_c.set_index('Symbol', inplace=True)

        # Use combine_first to patch metadata gaps from either file
        combined = df_n.combine_first(df_c)

        # 4. Final Rank Calculation (Additive Logic: SPY = 400)
        combined['rn_val'] = combined['rn_val'].fillna(0)
        combined['rc_val'] = combined['rc_val'].fillna(0)
        combined['rank_notional'] = combined['rn_val'] + combined['rc_val']

        # 5. Sort and Select Top 200
        final_universe = combined.sort_values(by='rank_notional', ascending=False).head(200)
        final_universe = final_universe.reset_index()

        # 6. Header Definition & Quote Enforcement
        target_columns = [
            "Symbol", "Name", "Price", "Pct Chg", "Trade Count",
            "Total $ Notional", "90-Day Avg $ Notional",
            "Relative Notional to 90-Day Avg", "Call $ Notional",
            "Put $ Notional", "Pct Single-Leg", "Pct Multi-Leg",
            "Pct Contingent", "rank_notional"
        ]

        # Ensure all requested columns exist
        for col in target_columns:
            if col not in final_universe.columns:
                final_universe[col] = np.nan

        # Final Export with double-quoted headers and strings
        final_universe[target_columns].to_csv(
            output_file,
            index=False,
            quoting=csv.QUOTE_NONNUMERIC
        )

        print(f"Success: {output_file} generated with quoted headers.")
        print(f"Sample Check: {final_universe[['Symbol', 'rank_notional']].head(3).to_dict('records')}")

    except Exception as e:
        print(f"Orchestration Error: {e}")

if __name__ == "__main__":
    generate_liquidity_eagle_universe()

Success: OptionVolume.csv generated with quoted headers.
Sample Check: [{'Symbol': 'SPY', 'rank_notional': 400.0}, {'Symbol': 'TSLA', 'rank_notional': 398.0}, {'Symbol': 'QQQ', 'rank_notional': 396.0}]
